In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['TF_GPU_THREAD_MODE'] = 'gpu_private'
os.environ['TF_GPU_THREAD_COUNT'] = '1'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0
from transformers import AutoTokenizer, TFBertModel

# Configuración básica
RUTA_CSV = "/home/esebas/Escritorio/Sebas/IA SAMSUNG/PROYECTO/datasets/text_and_description_dataset.csv"
RUTA_IMAGENES = "/home/esebas/Escritorio/Sebas/IA SAMSUNG/PROYECTO/extraccion memes/ALL MEMES"

MAX_LEN = 64
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 30
DROPOUT_RATE = 0.7
LEARNING_RATE = 1e-3
PATIENCE = 5

2026-02-02 18:32:20.451001: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/esebas/tf4050/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cargar y preparar datos
df = pd.read_csv(RUTA_CSV)
df['extracted_text'] = df['extracted_text'].fillna('')
df['description_es_new'] = df['description_es_new'].fillna('')

# Crear texto combinado para el modelo
df['text_final'] = df.apply(
    lambda row: f"MEME: {row['extracted_text'][:100]} [SEP] ESCENA: {row['description_es_new'][:150]}",
    axis=1
)

In [3]:
# Dividir datos en entrenamiento y validación
df_train, df_val = train_test_split(
    df, 
    test_size=0.20,
    random_state=42, 
    stratify=df['harmless']
)

In [4]:
# Tokenización con BERT español
tokenizer = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')

def tokenize_texts(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="tf"
    )

t_train = tokenize_texts(df_train['text_final'].tolist())
t_val = tokenize_texts(df_val['text_final'].tolist())

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
2026-02-02 18:32:27.929622: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1770053547.930580   61967 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [5]:
# Construcción del modelo ROBUST
def build_robust_model():
    img_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='img_input')
    
    # EfficientNet congelado para extracción visual
    cnn_base = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling='avg'
    )
    cnn_base.trainable = False
    
    visual_features = cnn_base(img_input)
    visual_proj = layers.Dense(64, activation='relu')(visual_features)
    visual_proj = layers.Dropout(DROPOUT_RATE)(visual_proj)
    
    ids_input = layers.Input(shape=(MAX_LEN,), dtype=tf.int32, name='ids_input')
    mask_input = layers.Input(shape=(MAX_LEN,), dtype=tf.int32, name='mask_input')
    
    # BERT congelado: usamos el pooler_output directamente
    bert_base = TFBertModel.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')
    bert_base.trainable = False
    
    bert_outputs = bert_base(ids_input, attention_mask=mask_input)
    textual_features = bert_outputs.pooler_output
    textual_proj = layers.Dense(64, activation='relu')(textual_features)
    textual_proj = layers.Dropout(DROPOUT_RATE)(textual_proj)
    
    # Fusión por concatenación simple
    merged = layers.Concatenate()([visual_proj, textual_proj])
    
    output = layers.Dense(1, activation='sigmoid', name='output')(merged)
    
    model = models.Model(inputs=[img_input, ids_input, mask_input], outputs=output)
    model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model

model = build_robust_model()

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing TFBertModel: ['mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFBertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert/pooler/dense/bias:0', 'bert/pooler/dense/kernel:0']
You should probably TRAIN this model on

In [6]:
# Función para procesar imágenes
def process_image(path):
    try:
        file = tf.io.read_file(path)
        img = tf.image.decode_jpeg(file, channels=3)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = tf.keras.applications.efficientnet.preprocess_input(img)
        return img
    except:
        # Imagen gris por si pasa algo 
        img = tf.ones((IMG_SIZE, IMG_SIZE, 3)) * 128
        return tf.keras.applications.efficientnet.preprocess_input(img)

In [7]:
# Preparar datasets
train_paths = [os.path.join(RUTA_IMAGENES, str(mid)) for mid in df_train['meme_id']]
val_paths = [os.path.join(RUTA_IMAGENES, str(mid)) for mid in df_val['meme_id']]

train_labels = df_train['harmless'].values.astype('float32')
val_labels = df_val['harmless'].values.astype('float32')

AUTOTUNE = tf.data.AUTOTUNE

In [8]:
# Dataset de entrenamiento
train_ds = tf.data.Dataset.from_tensor_slices((
    train_paths,
    t_train['input_ids'],
    t_train['attention_mask'],
    train_labels
)).shuffle(2000).map(
    lambda p, i, m, l: (
        {"img_input": process_image(p), "ids_input": i, "mask_input": m},
        l
    ),
    num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Dataset de validación
val_ds = tf.data.Dataset.from_tensor_slices((
    val_paths,
    t_val['input_ids'],
    t_val['attention_mask'],
    val_labels
)).map(
    lambda p, i, m, l: (
        {"img_input": process_image(p), "ids_input": i, "mask_input": m},
        l
    ),
    num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [9]:
# Balanceo de clases
y_train = df_train['harmless'].values
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

In [10]:
# Callbacks para entrenamiento
callbacks_list = [
    callbacks.ModelCheckpoint(
        filepath='./modelo_tiny_checkpoint.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_auc',
        patience=PATIENCE,
        min_delta=0.001,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    ),
]

In [11]:
# Entrenamiento
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=callbacks_list,
    verbose=1
)

Epoch 1/30


2026-02-02 18:32:43.271173: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-02-02 18:32:45.097587: I external/local_xla/xla/service/service.cc:163] XLA service 0x73774f114530 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-02 18:32:45.097608: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2026-02-02 18:32:45.102566: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1770053565.154230   62084 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


170/170 [==============================] - ETA: 0s - loss: 0.6751 - accuracy: 0.5903 - auc: 0.6248
Epoch 1: val_auc improved from -inf to 0.74758, saving model to ./modelo_tiny_checkpoint.keras


/home/esebas/tf4050/lib/python3.12/site-packages/transformers/generation/tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)


170/170 [==============================] - 44s 192ms/step - loss: 0.6751 - accuracy: 0.5903 - auc: 0.6248 - val_loss: 0.6189 - val_accuracy: 0.6750 - val_auc: 0.7476 - lr: 0.0010
Epoch 2/30
170/170 [==============================] - ETA: 0s - loss: 0.6196 - accuracy: 0.6518 - auc: 0.7130
Epoch 2: val_auc improved from 0.74758 to 0.76561, saving model to ./modelo_tiny_checkpoint.keras
170/170 [==============================] - 30s 175ms/step - loss: 0.6196 - accuracy: 0.6518 - auc: 0.7130 - val_loss: 0.5915 - val_accuracy: 0.6882 - val_auc: 0.7656 - lr: 0.0010
Epoch 3/30
170/170 [==============================] - ETA: 0s - loss: 0.5936 - accuracy: 0.6759 - auc: 0.7462
Epoch 3: val_auc improved from 0.76561 to 0.78048, saving model to ./modelo_tiny_checkpoint.keras
170/170 [==============================] - 30s 178ms/step - loss: 0.5936 - accuracy: 0.6759 - auc: 0.7462 - val_loss: 0.5674 - val_accuracy: 0.7176 - val_auc: 0.7805 - lr: 0.0010
Epoch 4/30
170/170 [===========================

In [12]:
if 'val_auc' in history.history:
    best_epoch = np.argmax(history.history['val_auc'])
    best_val_auc = history.history['val_auc'][best_epoch]
    best_val_acc = history.history['val_accuracy'][best_epoch]
    best_train_acc = history.history['accuracy'][best_epoch]
    
    train_val_gap = best_train_acc - best_val_acc
    
    print(f"Mejor época: {best_epoch + 1}")
    print(f"Train Accuracy: {best_train_acc:.4f}")
    print(f"Val Accuracy: {best_val_acc:.4f}")
    print(f"Val AUC: {best_val_auc:.4f}")
    print(f"Gap train-val: {train_val_gap*100:.1f}%")

Mejor época: 10
Train Accuracy: 0.7800
Val Accuracy: 0.7257
Val AUC: 0.8091
Gap train-val: 5.4%


In [ ]:
model.save('./modelo_memes4good_ROBUST.keras')